## Instalar modulos adicionales

Antes de poder hacer la extraccion de los datos es necesario garantizar que el modulo **playwright** este instaldo dentro de python.

In [ ]:
# instalar modulo en python
!pip install playwright
# instalar componentes en el server
!playwright install
# preparar el ambiente
!apt-get update
!apt-get install -y libnss3 libatk-bridge2.0-0 libgtk-3-0 libgbm-dev

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 12.6 MB/s eta 0:00:00
(node:5857) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
167.3 MiB [] 0% 340.0s167.3 MiB [] 0% 23.4s167.3 MiB [] 0% 19.0s167.3 MiB [] 0% 14.6s167.3 MiB [] 0% 7.8s167.3 MiB [] 1% 4.5s167.3 MiB [] 2% 4.3s167.3 MiB [] 3% 3.4s167.3 MiB [] 3% 3.3s167.3 MiB [] 4% 3.2s167.3 MiB [] 5% 2.7s167.3 MiB [] 7% 2.3s167.3 MiB [] 8% 2.1s167.3 MiB [] 9% 2.1s167.3 MiB [] 10% 2.0s167.3 MiB [] 12% 1.8s167.3 MiB [] 13% 1.8s167.3 MiB [] 14% 1.7s167.3 MiB [] 15% 1.8s167.3 MiB [] 16% 1.8s167.3 MiB [] 17% 1.8s167.3 MiB [] 18% 1.7s167.3 MiB [] 19% 1.8s167.3 MiB [] 20% 1.8s167.3 MiB [] 21% 1.8s167.3 MiB [] 22% 1.8s167.3 MiB [] 23% 1.8s167.3 MiB [] 24% 1.8s167.3 MiB [] 25% 1.8s167.3 MiB [] 26% 1.7s

## Script para tomar los datos

In [ ]:
# importar los modulos
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup

In [ ]:
# definir las variables
URL="https://www.metrocuadrado.com/apartamento-apartaestudio/arriendo/bogota/"

In [ ]:
# hacer la solicitud a la pagina y guardar los datos
async def main():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=["--no-sandbox"])
        page = await browser.new_page()

        await page.goto(URL)

        await page.wait_for_selector(".property-card__container")

        previous_count = 0

        # 🔁 hacer scroll varias veces
        for _ in range(10):
            cards = await page.query_selector_all(".property-card__container")
            current_count = len(cards)

            print("Cargados:", current_count)

            if current_count == previous_count:
                break  # no cargó más

            previous_count = current_count

            # scroll hacia abajo
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")

            # esperar que cargue más contenido
            await page.wait_for_timeout(2000)

        # 🧠 ahora sí extraer todo
        html = await page.content()
        soup = BeautifulSoup(html, "html.parser")

        propiedades = soup.find_all("div", class_="property-card__container")

        print("\nTotal final:", len(propiedades))

        with open("pagina_1.html", "w", encoding="utf-8") as f:
          f.write(html)

        await browser.close()

await main()

Cargados: 6
Cargados: 21
Cargados: 68
Cargados: 68

Total final: 68


In [ ]:
# lee los datos y valida que esten
with open("pagina_1.html", "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "html.parser")
cards = soup.find_all("div", class_="property-card__container")

print("Total de cards:", len(cards))

Total de cards: 68


## Script que poder esquematizar los datos
con un solo dato, el de la posicion 0   

In [ ]:
# imprimir el elemento 1
cards[0]

<div class="property-card__container property-card__highlighted"><div class="property-card__content"><a href="/inmueble/arriendo-apartamento-bogota-chico-norte-1-habitaciones-1-banos/21282-M6099315?src_url=%2Fapartamento-apartaestudio%2Farriendo%2Fbogota%2F" rel="noreferrer" target="_blank"><div class="property-card__photo"><img alt="Foto de Apartamento en Arriendo en CHICO NORTE, Bogotá D.C. con 1 habitaciones, 1 baños, área 38 m2 - 21282-M6099315" data-nimg="1" decoding="async" height="217.33" src="https://multimedia.metrocuadrado.com/21282-M6099315/21282-M6099315_20_p.jpg" style="color: transparent;" title="Apartamento en Arriendo en CHICO NORTE, Bogotá D.C. - 21282-M6099315" width="324"/><div class="property-card__photo-tags"><pt-tag big-type="clientes" class="hydrated" element-id="higlightTag" icon="crown" size="small" type="blue">Destacado</pt-tag></div></div><div class="property-card__detail"><div class="property-card__detail-top"><div class="property-card__detail-top__left"><di

In [ ]:
#href = cards[0].find_all("div", class_="property-card__content")[0].find_all("a")[0]["href"]
href = cards[0].find_all("a")[0]["href"]
href

'/inmueble/arriendo-apartamento-bogota-chico-norte-1-habitaciones-1-banos/21282-M6099315?src_url=%2Fapartamento-apartaestudio%2Farriendo%2Fbogota%2F'

In [ ]:
url_img = cards[0].find_all("img")[0]["src"]
url_img

'https://multimedia.metrocuadrado.com/21282-M6099315/21282-M6099315_20_p.jpg'

In [ ]:
alt_img = cards[0].find_all("img")[0]["alt"]
alt_img

'Foto de Apartamento en Arriendo en CHICO NORTE, Bogotá D.C. con 1 habitaciones, 1 baños, área 38 m2 - 21282-M6099315'

In [14]:
barrio = cards[0].find_all("div", class_="property-card__detail-top__left")[0].find_all("div")[0]
barrio

<div>CHICO NORTE | Norte | Bogotá D.C.</div>

In [ ]:
import requests
response = requests.get(URL)

In [ ]:
response

<Response [200]>